# 04. 집중호우 사례 분석 (Case Study: Heavy Rain)

## 목표
- 집중호우 주간(9/17~9/23) vs 이전 주간(9/10~9/16) 비교
- 시간대별, 대여소별 변화 분석
- 정비·점검 효율화 주간 근거 도출

## ULTRA-THINK Framework: R - Reveal Insights (인사이트 도출)

In [ ]:
import sys
import os
sys.path.append(os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.utils.preprocessing import filter_heavy_rain_period
from src.utils.visualization import plot_comparison_before_after

import warnings
warnings.filterwarnings('ignore')

## 1. 데이터 로딩 및 기간 필터링

In [ ]:
# 전처리된 데이터 로딩
df = pd.read_csv("../data/processed/merged_hourly_202509.csv", encoding="utf-8")
df['기준_날짜'] = pd.to_datetime(df['기준_날짜'])

print(f"전체 데이터: {len(df):,}개 레코드")

In [ ]:
# 기간 필터링
before_period = df[(df['기준_날짜'] >= '2025-09-10') & (df['기준_날짜'] <= '2025-09-16')].copy()
heavy_rain_period = filter_heavy_rain_period(df, start_date='2025-09-17', end_date='2025-09-23')

print(f"\n이전 주간 (9/10~9/16): {len(before_period):,}개 레코드")
print(f"집중호우 주간 (9/17~9/23): {len(heavy_rain_period):,}개 레코드")

## 2. 전후 비교: 총 이용량

In [ ]:
# 총 이용량 비교
total_before = before_period['전체_건수'].sum()
total_heavy_rain = heavy_rain_period['전체_건수'].sum()
change_pct = ((total_heavy_rain - total_before) / total_before) * 100

print(f"\n=== 총 이용량 비교 ===")
print(f"이전 주간: {total_before:,} 건")
print(f"집중호우 주간: {total_heavy_rain:,} 건")
print(f"변화율: {change_pct:+.2f}%")

In [ ]:
# 시각화
fig, ax = plt.subplots(figsize=(8, 6))
bars = ax.bar(['이전 주간\n(9/10~9/16)', '집중호우 주간\n(9/17~9/23)'], 
               [total_before, total_heavy_rain], 
               color=['#66C2A5', '#FC8D62'])
ax.set_ylabel('총 이용 건수', fontsize=12)
ax.set_title('집중호우 전후 총 이용량 비교', fontsize=14, fontweight='bold')
ax.grid(axis='y', alpha=0.3)

for bar in bars:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{int(height):,}',
            ha='center', va='bottom', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.savefig('../outputs/figures/total_usage_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

## 3. 전후 비교: 시간대별 / 요일별

In [ ]:
# 파생 변수 추가
before_period['요일'] = before_period['기준_날짜'].dt.dayofweek
heavy_rain_period['요일'] = heavy_rain_period['기준_날짜'].dt.dayofweek

# 시각화
plot_comparison_before_after(
    before_df=before_period,
    after_df=heavy_rain_period,
    save_path="../outputs/figures/comparison_heavy_rain.png"
)

## 4. 시간대별 감소율 분석

In [ ]:
# 시간대별 평균 이용량
before_hourly = before_period.groupby('기준_시간')['전체_건수'].mean()
heavy_rain_hourly = heavy_rain_period.groupby('기준_시간')['전체_건수'].mean()

# 감소율 계산
hourly_change = ((heavy_rain_hourly - before_hourly) / before_hourly * 100)

print("\n=== 시간대별 변화율 TOP 5 (가장 많이 감소) ===")
print(hourly_change.sort_values().head(5))

print("\n=== 시간대별 변화율 LOW 5 (가장 적게 감소) ===")
print(hourly_change.sort_values().tail(5))

## 5. 대여소별 영향 분석

In [ ]:
# 대여소별 평균 이용량
before_station = before_period.groupby('시작_대여소_ID')['전체_건수'].mean()
heavy_rain_station = heavy_rain_period.groupby('시작_대여소_ID')['전체_건수'].mean()

# 감소율 계산
station_change = ((heavy_rain_station - before_station) / before_station * 100).dropna()

print("\n=== 대여소별 감소율 TOP 10 ===")
print(station_change.sort_values().head(10))

In [ ]:
# 시각화
top_affected = station_change.sort_values().head(15)

plt.figure(figsize=(10, 8))
plt.barh(range(len(top_affected)), top_affected.values, color='#E63946')
plt.yticks(range(len(top_affected)), top_affected.index)
plt.xlabel('이용량 변화율 (%)', fontsize=12)
plt.ylabel('대여소 ID', fontsize=12)
plt.title('집중호우 기간 영향을 가장 많이 받은 대여소 TOP 15', fontsize=14, fontweight='bold')
plt.axvline(0, color='black', linewidth=1, linestyle='--')
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig('../outputs/figures/top_affected_stations.png', dpi=300, bbox_inches='tight')
plt.show()

## 6. 인사이트 요약

In [ ]:
insights = f"""
=== 집중호우 주간 분석 인사이트 ===

1. 총 이용량 변화
   - 집중호우 주간 총 이용량: {total_heavy_rain:,} 건
   - 이전 주 대비: {change_pct:+.1f}%

2. 시간대별 영향
   - 가장 큰 감소: {hourly_change.idxmin()}시 ({hourly_change.min():.1f}%)
   - 평균 감소율: {hourly_change.mean():.1f}%

3. 대여소별 영향
   - 가장 영향 받은 대여소: {station_change.idxmin()} ({station_change.min():.1f}%)
   - 영향 받은 대여소 수: {len(station_change[station_change < -20])}개 (20% 이상 감소)

4. 정비·점검 효율화 주간 근거
   - 집중호우 기간 이용량 급감으로 정비·점검 집중 투입에 최적
   - 운영 공백 최소화 및 생산성 향상 가능
"""

print(insights)

# 인사이트 저장
with open('../outputs/reports/heavy_rain_insights.txt', 'w', encoding='utf-8') as f:
    f.write(insights)

print("\n✅ 인사이트 저장 완료: outputs/reports/heavy_rain_insights.txt")

## 7. 다음 단계

- ✅ 집중호우 사례 분석 완료
- ✅ 전후 비교 및 인사이트 도출 완료
- 다음: `05_insights_and_actions.ipynb`에서 최종 인사이트 및 실행 방안 정리